In [ ]:
from typing import Type
from langchain_openai import ChatOpenAI
from langchain.tools import BaseTool
from langchain_core.messages.ai import AIMessage
from langchain.agents import create_agent
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_community.document_loaders.web_base import WebBaseLoader
from pydantic import BaseModel, Field
import os, openai

In [18]:
conversation = openai.conversations.create(
    items=[{"role": "user", "content": "what are the 5 Ds of dodgeball?"}],
    metadata={"user_id": "peter_le_fleur"},
)

print(conversation)

Conversation(id='conv_6990b28f33148194bfe67a3b05dbeaae01d71d0692ad031c', created_at=1771090575, metadata={'user_id': 'peter_le_fleur'}, object='conversation')


In [31]:
response = openai.responses.create(
    model="gpt-4.1",
    input=[{"role": "user", "content": "What are the 5 Ds of dodgeball?"}],
    conversation=conversation.id,
)

print(response)

Response(id='resp_01d71d0692ad031c006990bf0153a081949300c91b787ced4f', created_at=1771093761.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_01d71d0692ad031c006990bf0227f8819486cde6e4d5b94a81', content=[ResponseOutputText(annotations=[], text='The **5 Ds of Dodgeball** are:\n\n1. **Dodge**\n2. **Duck**\n3. **Dip**\n4. **Dive**\n5. **Dodge**\n\nThese became popular from the movie **"Dodgeball: A True Underdog Story,"** where the character Patches O\'Houlihan says, “If you can dodge a wrench, you can dodge a ball!” The phrase is a humorous way to teach the importance of agility and quick movements while playing dodgeball.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1771093763.0, conversation=Conversation(id='conv_6990b2

In [32]:
for output in response.output:
    for content in output.content:
        print(content.text)

The **5 Ds of Dodgeball** are:

1. **Dodge**
2. **Duck**
3. **Dip**
4. **Dive**
5. **Dodge**

These became popular from the movie **"Dodgeball: A True Underdog Story,"** where the character Patches O'Houlihan says, “If you can dodge a wrench, you can dodge a ball!” The phrase is a humorous way to teach the importance of agility and quick movements while playing dodgeball.


In [33]:
print(response.output[0].content[0].text)

The **5 Ds of Dodgeball** are:

1. **Dodge**
2. **Duck**
3. **Dip**
4. **Dive**
5. **Dodge**

These became popular from the movie **"Dodgeball: A True Underdog Story,"** where the character Patches O'Houlihan says, “If you can dodge a wrench, you can dodge a ball!” The phrase is a humorous way to teach the importance of agility and quick movements while playing dodgeball.


In [34]:
print(response.output_text)

The **5 Ds of Dodgeball** are:

1. **Dodge**
2. **Duck**
3. **Dip**
4. **Dive**
5. **Dodge**

These became popular from the movie **"Dodgeball: A True Underdog Story,"** where the character Patches O'Houlihan says, “If you can dodge a wrench, you can dodge a ball!” The phrase is a humorous way to teach the importance of agility and quick movements while playing dodgeball.


In [12]:
retrieved_response = openai.responses.retrieve(response_id=response.id)
print(retrieved_response)

Response(id='resp_0030e0074eb178d6006990a7562e6c8190b07f233e455ed6e6', created_at=1771087702.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_0030e0074eb178d6006990a75705ec8190b8aa8199dc5cf339', content=[ResponseOutputText(annotations=[], text='The **5 Ds of dodgeball** are:\n\n1. **Dodge**\n2. **Duck**\n3. **Dip**\n4. **Dive**\n5. **Dodge** (again)\n\nThis humorous phrase comes from the movie **"Dodgeball: A True Underdog Story"**, where the character Patches O\'Houlihan teaches it as the key to success in dodgeball.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1771087703.0, conversation=Conversation(id='conv_6990a6bab37c8190822a9a1cc4b17e480030e0074eb178d6'), max_output_tokens=None, max_tool_calls=None, previous_respo

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },
            "required": ["sign"],
        },
    },
]


def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."


# Create a running input list we will add to over time
input_list = [{"role": "user", "content": "What is my horoscope? I am an Aquarius."}]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)

# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_horoscope":
            # 3. Execute the function logic for get_horoscope
            horoscope = get_horoscope(json.loads(item.arguments))

            # 4. Provide function call results to the model
            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps({"horoscope": horoscope}),
                }
            )

print("Final input:")
print(input_list)

response = client.responses.create(
    model="gpt-5",
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final input:
[{'role': 'user', 'content': 'What is my horoscope? I am an Aquarius.'}, ResponseReasoningItem(id='rs_04600a429137c203006991b89b50788194b04f94b57dc84f09', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"sign":"Aquarius"}', call_id='call_chHPfVEpkohI1cJPCAGGJugF', name='get_horoscope', type='function_call', id='fc_04600a429137c203006991b89da7e08194828376d84a2f3c86', status='completed'), {'type': 'function_call_output', 'call_id': 'call_chHPfVEpkohI1cJPCAGGJugF', 'output': '{"horoscope": "{\'sign\': \'Aquarius\'}: Next Tuesday you will befriend a baby otter."}'}]
Final output:
{
  "id": "resp_04600a429137c203006991b89ead808194b25da84660d60821",
  "created_at": 1771157662.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a horoscope generated by a tool.",
  "metadata": {},
  "model": "gpt-5-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "rs_04600a